# Figure 3C and Figure S13 — cell-type specificity of the top LASSO transcripts

Dot plots of mean expression (colour, scaled 0–1 per gene) and fraction of expressing cells (size) for
the top 10 LASSO-selected protein-coding transcripts (Figure 3B) across cell types of three scRNA-seq atlases.

**Inputs** (`results/scrna/`, written by `00_scrna_prep/01_subset_dotplot_atlases.ipynb`):
`tabula_sapiens_dotplot.h5ad`, `strati_dotplot.h5ad`, `li2025_dotplot.h5ad`.
Each holds log-normalised `.X` for the 10 genes and the cell-type column, whose (ordered) categories give the plotting order.

**Outputs** (`plots/`):

| Panel | Atlas | File |
|---|---|---|
| Figure 3C (also Figure S13A) | Tabula Sapiens v2, healthy tissues | `Fig3C_dotplot_tabula_sapiens.pdf` / `.png` |
| Figure S13B | Strati et al. 2023, B-NHL bone marrow mononuclear cells | `SFig13B_dotplot_strati_bmmc.pdf` |
| Figure S13C | Li et al. 2025, LBCL microenvironment (tumour cells excluded) | `SFig13C_dotplot_li2025_lbcl_tme.pdf` |

In [ ]:
import sys; sys.path.insert(0, "../../utils")
from config import res_in, plot_path

import anndata as ad
import matplotlib.pyplot as plt
import scanpy as sc

tabsap = ad.read_h5ad(res_in("scrna", "tabula_sapiens_dotplot.h5ad"))
strati = ad.read_h5ad(res_in("scrna", "strati_dotplot.h5ad"))
li2025 = ad.read_h5ad(res_in("scrna", "li2025_dotplot.h5ad"))

In [ ]:
# Top 10 LASSO-selected protein-coding transcripts (Figure 3B), in plotting order
GENE_LIST = ["IGHD", "IGLC3", "MS4A1", "TDRD9", "SPDYC", "CLC", "ACCS", "FHDC1", "RGS16", "ALOX15"]

# Double-column print style (7 inch width)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
})


def cell_type_dotplot(adata, groupby, name, png=False):
    """Genes (rows) x cell types (columns) dot plot, saved to plots/<name>.pdf (and .png)."""
    sc.pl.dotplot(
        adata,
        var_names=GENE_LIST,
        gene_symbols="feature_name",
        groupby=groupby,
        categories_order=list(adata.obs[groupby].cat.categories),
        standard_scale="var",
        cmap="Reds",
        use_raw=False,
        swap_axes=True,
        dendrogram=False,
        figsize=(7, 2.35),
        show=False,
    )
    # rotate the cell-type labels of the main panel (first axis; the others are legends)
    ax_main = plt.gcf().get_axes()[0]
    ax_main.set_xticklabels(ax_main.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    plt.tight_layout()

    plt.savefig(plot_path(f"{name}.pdf"), bbox_inches="tight", dpi=300, facecolor="white", edgecolor="none")
    if png:
        plt.savefig(plot_path(f"{name}.png"), bbox_inches="tight", dpi=300, facecolor="white")
    plt.show()

## Figure 3C — Tabula Sapiens (also Figure S13A)
Broad cell classes are shown with a capitalised first letter.

In [ ]:
tabsap.obs["broad_cell_class_label"] = tabsap.obs["broad_cell_class"].cat.rename_categories(
    lambda c: (c[0].upper() + c[1:]).replace("of b lineage", "of B lineage")
)
cell_type_dotplot(tabsap, "broad_cell_class_label", "Fig3C_dotplot_tabula_sapiens", png=True)

## Figure S13B — Strati et al. 2023, B-NHL bone marrow

In [ ]:
cell_type_dotplot(strati, "cell_type", "SFig13B_dotplot_strati_bmmc")

## Figure S13C — Li et al. 2025, LBCL tumour microenvironment

In [ ]:
cell_type_dotplot(li2025, "cell_type", "SFig13C_dotplot_li2025_lbcl_tme")